Connected to reactive-agent (3.11.x) (Python 3.11.-1)

 # Input validation

 **First node every message hits** — blocks injections and sets the risk level
 before the LLM sees anything.

 Two-stage evaluation:
 - **Stage 1 — hard block:** injection patterns trigger an immediate return.
   The LLM never runs, no tokens consumed.
 - **Stage 2 — risk scoring:** high-risk keywords don't block — they set
   `risk_level: "high"` and `requires_human_approval: True`.
   The request proceeds but routes through `human_loop` before any consequential tool runs.

In [ ]:
import re
from app.agent.state import AgentState
from app.core.logging import get_logger
import time

log = get_logger(__name__)

 ## Patterns and keywords

 Patterns are compiled once at module load — not on every request.
 At call time: `pattern.search(content_lower)` on the compiled object.
 The bug to avoid: `re.search(pattern, text)` treats the compiled pattern
 as a string and raises a `TypeError`.

 `re.IGNORECASE` is set on the patterns, and `content_lower` is already
 lowercased — the flag is redundant but harmless.
 The tradeoff: lowercasing loses original casing, which matters if you ever
 want to log the exact matched substring.

In [ ]:
_BLOCKED_PATTERNS_RAW = [
    r"ignore (all|previous|above) instructions",
    r"jailbreak",
    r"act as (DAN|an AI without restrictions)",
    r"bypass (safety|guardrails|filters)",
    r"you are now",
    r"pretend (you are|to be)",
    r"prompt injection",
]

BLOCKED_PATTERNS = [re.compile(p, re.IGNORECASE) for p in _BLOCKED_PATTERNS_RAW]

HIGH_RISK_KEYWORDS = [
    "remove", "erase", "delete",
    "drop table", "all data", "all users",
    "bank transfer", "transfer", "payment",
]

MAX_INPUT_LENGTH = 4000

 ## `input_validation_node`

 **Return shape on pass:** initializes `input_validated`, `blocked`, `risk_level`,
 `requires_human_approval`, `iterations`, `max_iterations`, `workflow_trace`.
 This node runs first — it's responsible for setting up these fields
 before any downstream node reads them.

 The `break` after the first keyword match is intentional —
 one keyword is enough to flag high risk, no need to scan further.

In [ ]:
async def input_validation_node(state: AgentState) -> dict:
    messages = state.get("messages", [])
    if not messages:
        return {"blocked": True, "block_reason": "No message received"}

    last_message = messages[-1]
    content = last_message.content if hasattr(last_message, "content") else str(last_message)

    if len(content) > MAX_INPUT_LENGTH:
        return {
            "blocked": True,
            "block_reason": f"Content too large: {len(content)} chars (max {MAX_INPUT_LENGTH})",
        }

    content_lower = content.lower()

    for pattern in BLOCKED_PATTERNS:
        if pattern.search(content_lower):
            log.warning("prompt_injection_detected: pattern=%s", pattern.pattern)
            return {
                "blocked": True,
                "block_reason": "Prompt injection detected",
            }

    risk_level = "low"
    requires_human = False
    for keyword in HIGH_RISK_KEYWORDS:
        if keyword in content_lower:
            risk_level = "high"
            requires_human = True
            log.warning("high_risk_keyword: keyword=%s", keyword)
            break

    workflow_trace = state.get("workflow_trace", [])
    workflow_trace.append({
        "node": "input_guard",
        "status": "passed",
        "risk_level": risk_level,
        "timestamp": time.time(),
    })

    return {
        "input_validated": True,
        "blocked": False,
        "risk_level": risk_level,
        "requires_human_approval": requires_human,
        "iterations": state.get("iterations", 0),
        "max_iterations": state.get("max_iterations", 10),
        "workflow_trace": workflow_trace,
    }